# Module 6: r,w, and n Gairaigo Game
---

In [1]:
# Katakana Gairaigo Puzzle — Module 6 (R-row + ワ／ヲ／ン)
# Build each word by clicking katakana pieces. ✔ Check to verify; ▶ Play for TTS (ja-JP).
# Uses string.Template to avoid f-string interpolation issues.

from IPython.display import HTML, display
from textwrap import dedent
from string import Template
import uuid, json, random

uid = "kata_puzzle_mod6_" + uuid.uuid4().hex[:8]

# ==============================
# WORDS (Module 6 vocab + onomatopoeia)
# ==============================
WORDS = [
    {"en":"iron",      "romaji":"airon",       "jp":"アイロン"},
    {"en":"ink",       "romaji":"inku",        "jp":"インク"},
    {"en":"coin",      "romaji":"koin",        "jp":"コイン"},
    {"en":"wieners",   "romaji":"uinnā",       "jp":"ウインナー"},
    {"en":"guitar",    "romaji":"gitā",        "jp":"ギター"},
    {"en":"okra",      "romaji":"okura",       "jp":"オクラ"},
    {"en":"lion",      "romaji":"raion",       "jp":"ライオン"},
    {"en":"camera",    "romaji":"kamera",      "jp":"カメラ"},
    {"en":"koala",     "romaji":"koara",       "jp":"コアラ"},
    {"en":"salad",     "romaji":"sarada",      "jp":"サラダ"},
    {"en":"slippers",  "romaji":"surippa",     "jp":"スリッパ"},
    {"en":"broccoli",  "romaji":"burokkorī",   "jp":"ブロッコリー"},
    {"en":"towel",     "romaji":"taoru",       "jp":"タオル"},
    {"en":"bench",     "romaji":"benchi",      "jp":"ベンチ"},
    {"en":"TV",        "romaji":"terebi",      "jp":"テレビ"},
    {"en":"tent",      "romaji":"tento",       "jp":"テント"},
    {"en":"toilet",    "romaji":"toire",       "jp":"トイレ"},
    {"en":"ramen",     "romaji":"rāmen",       "jp":"ラーメン"},
    {"en":"hint",      "romaji":"hinto",       "jp":"ヒント"},
    {"en":"hairpin",   "romaji":"heapin",      "jp":"ヘアピン"},
    {"en":"milk",      "romaji":"miruku",      "jp":"ミルク"},
    {"en":"medal",     "romaji":"medaru",      "jp":"メダル"},
    {"en":"melon",     "romaji":"meron",       "jp":"メロン"},
    {"en":"model",     "romaji":"moderu",      "jp":"モデル"},
    {"en":"earring",   "romaji":"iyaringu",    "jp":"イヤリング"},
    {"en":"yogurt",    "romaji":"yōguruto",    "jp":"ヨーグルト"},
    {"en":"lamp",      "romaji":"ranpu",       "jp":"ランプ"},
    {"en":"ball",      "romaji":"bōru",        "jp":"ボール"},
    {"en":"lettuce",   "romaji":"retasu",      "jp":"レタス"},
    {"en":"lemon",     "romaji":"remon",       "jp":"レモン"},
    {"en":"rope",      "romaji":"rōpu",        "jp":"ロープ"},
    {"en":"rocket",    "romaji":"roketto",     "jp":"ロケット"},
    {"en":"wine",      "romaji":"wain",        "jp":"ワイン"},
    {"en":"shower",    "romaji":"shawā",       "jp":"シャワー"},
    {"en":"jeans",     "romaji":"jīnzu",       "jp":"ジーンズ"},
    {"en":"dance",     "romaji":"dansu",       "jp":"ダンス"},
    {"en":"panda",     "romaji":"panda",       "jp":"パンダ"},
    {"en":"pudding",   "romaji":"purin",       "jp":"プリン"},
    # Onomatopoeia
    {"en":"rumble / purr (sfx)",     "romaji":"gorogoro",  "jp":"ゴロゴロ"},
    {"en":"clattering / rattle (sfx)","romaji":"garagara", "jp":"ガラガラ"},
    {"en":"woof-woof (dog)",         "romaji":"wanwan",    "jp":"ワンワン"},
    {"en":"excited / thrilled (sfx)","romaji":"wakuwaku",  "jp":"ワクワク"},
    {"en":"falling drops/crumbs",    "romaji":"poroporo",  "jp":"ポロポロ"},
    {"en":"flapping / hectic",       "romaji":"batabata",  "jp":"バタバタ"},
]

# Build the master pool of katakana pieces used, plus focused decoys
ALL_SET = set(ch for w in WORDS for ch in w["jp"])
# Add common pieces & diacritics, long vowel mark, sokuon, small kana
DECOYS = (
    "アイウエオカキクケコサシスセソタチツテトナニヌネノハヒフヘホマミムメモ"
    "ヤユヨラリルレロワヲン"
    "ガギグゲゴザジズゼゾダヂヅデドバビブベボパピプペポ"
    "ァィゥェォャュョッーヴ"
)
for ch in DECOYS:
    ALL_SET.add(ch)
PIECES = sorted(ALL_SET)

root_id = f"gp-root-{uid}"

tpl = Template(dedent("""
<div id="$root_id" style="font-family: system-ui, -apple-system, Segoe UI, Roboto, Helvetica, Arial;">
  <style>
    #$root_id h2 { margin: 8px 0 4px; }
    #$root_id .row { display:flex; gap:10px; align-items:center; flex-wrap:wrap; }
    #$root_id .panel { border:1px solid #e2e2e2; border-radius:12px; padding:10px; background:#fff; }
    #$root_id .prompt { display:flex; flex-direction:column; gap:4px; }
    #$root_id .prompt .en { font-size:1.15rem; font-weight:600; }
    #$root_id .prompt .romaji { color:#555; }
    #$root_id .target { min-height:56px; display:flex; gap:6px; align-items:center; flex-wrap:wrap; border:1px dashed #bbb; border-radius:10px; padding:8px; background:#fafafa; }
    #$root_id .target .char { font-size:1.6rem; padding:4px 8px; border-radius:8px; background:#f0f3ff; border:1px solid #b7c5ff; }
    #$root_id .controls button, #$root_id .pool button { cursor:pointer; border-radius:10px; border:1px solid #ccc; background:#fff; padding:8px 10px; }
    #$root_id .controls button:hover, #$root_id .pool button:hover { background:#f6f6f6; }
    #$root_id .pool { display:grid; grid-template-columns: repeat(auto-fill, minmax(46px, 1fr)); gap:6px; }
    #$root_id .pool .piece { font-size:1.2rem; padding:8px 0; }
    #$root_id .status { font-size:.95rem; color:#444; min-height:1.2em; }
    #$root_id .ok { color:#0a8a0a; }
    #$root_id .bad { color:#b00020; }
    #$root_id .hint { color:#666; font-size:.92rem; }
    #$root_id .ratebox { margin-left:10px; display:flex; align-items:center; gap:6px; }
    #$root_id .voiceinfo { font-size:.9rem; color:#444; }
  </style>

  <h2>🧩 Katakana Gairaigo Puzzle — Module 6 (R + ワ／ヲ／ン)</h2>
  <div class="panel">
    <div class="row">
      <div class="prompt">
        <div class="en" id="gp-board-$uid">—</div>
        <div class="romaji hint" id="gp-romaji-$uid">—</div>
      </div>
      <div class="row" style="margin-left:auto;">
        <span class="voiceinfo" id="gp-voiceinfo-$uid">Voice: (detecting Google Japanese…)</span>
        <div class="ratebox">
          <label for="gp-rate-$uid"><b>Speed:</b></label>
          <input type="range" id="gp-rate-$uid" min="0.7" max="1.5" step="0.05" value="1.00">
          <span id="gp-rateval-$uid">1.00×</span>
        </div>
      </div>
    </div>

    <div id="gp-target-$uid" class="target" aria-live="polite"></div>

    <div class="row controls" style="margin-top:10px;">
      <button id="gp-check-$uid">✔ Check</button>
      <button id="gp-play-$uid">▶ Play</button>
      <button id="gp-back-$uid">⌫ Backspace</button>
      <button id="gp-clear-$uid">🧹 Clear</button>
      <button id="gp-shuffle-$uid">🔀 Shuffle</button>
      <button id="gp-new-$uid">🎲 New word</button>
    </div>
    <div id="gp-status-$uid" class="status"></div>
  </div>

  <div class="panel" style="margin-top:12px;">
    <div class="hint" style="margin-bottom:6px;">
      Build the katakana for the loanword. Use ッ for doubled consonants and ー for long vowels when needed.
    </div>
    <div id="gp-pool-$uid" class="pool"></div>
  </div>

  <script>
  (function(){
    const WORDS = $WORDS_JSON;
    const ALL   = $PIECES_JSON;

    const boardEl   = document.getElementById("gp-board-$uid");
    const romajiEl  = document.getElementById("gp-romaji-$uid");
    const targetEl  = document.getElementById("gp-target-$uid");
    const poolEl    = document.getElementById("gp-pool-$uid");
    const statusEl  = document.getElementById("gp-status-$uid");
    const newBtn    = document.getElementById("gp-new-$uid");
    const checkBtn  = document.getElementById("gp-check-$uid");
    const clearBtn  = document.getElementById("gp-clear-$uid");
    const backBtn   = document.getElementById("gp-back-$uid");
    const shuffleBtn= document.getElementById("gp-shuffle-$uid");
    const playBtn   = document.getElementById("gp-play-$uid");
    const voiceInfo = document.getElementById("gp-voiceinfo-$uid");
    const rate      = document.getElementById("gp-rate-$uid");
    const rateVal   = document.getElementById("gp-rateval-$uid");

    let currentVoice = null;
    let current = null;
    let answer = [];

    function updateRateLabel(){
      const r = parseFloat(rate.value) || 1.0;
      rateVal.textContent = r.toFixed(2) + "×";
    }
    rate.addEventListener("input", updateRateLabel);
    updateRateLabel();

    function pickGoogleJa(list){
      const lower = s => (s||"").toLowerCase();
      let v = list.find(v => lower(v.name).includes("google") && v.lang && v.lang.toLowerCase().startsWith("ja"));
      if (v) return v;
      v = list.find(v => v.lang && v.lang.toLowerCase().startsWith("ja"));
      return v || list[0] || null;
    }
    function describe(v){ return v ? "Voice: " + (v.name||"?") + " — " + (v.lang||"?") : "Voice: (none)"; }

    function tryLoadVoicesOnce(){
      const list = speechSynthesis.getVoices() || [];
      if (list.length){
        currentVoice = pickGoogleJa(list);
        voiceInfo.textContent = describe(currentVoice);
        return true;
      }
      return false;
    }
    function ensureVoice(){
      if (tryLoadVoicesOnce()) return;
      try { const warm = new SpeechSynthesisUtterance(" "); warm.volume=0; speechSynthesis.speak(warm); } catch(e){}
      let tries = 0;
      const t = setInterval(()=>{
        tries++;
        if (tryLoadVoicesOnce() || tries>12) clearInterval(t);
      },200);
    }
    if (speechSynthesis.addEventListener){
      speechSynthesis.addEventListener("voiceschanged", tryLoadVoicesOnce);
    } else {
      speechSynthesis.onvoiceschanged = tryLoadVoicesOnce;
    }
    ensureVoice();

    function currentRate(){
      const r = parseFloat(rate.value);
      return Math.max(0.7, Math.min(1.5, isNaN(r)?1.0:r));
    }
    function speakJa(text){
      const t = (text||"").trim();
      if (!t) return;
      const u = new SpeechSynthesisUtterance(t);
      u.lang = "ja-JP";
      if (currentVoice) u.voice = currentVoice;
      u.rate = currentRate();
      speechSynthesis.cancel();
      speechSynthesis.speak(u);
    }

    function choice(arr){ return arr[Math.floor(Math.random()*arr.length)]; }
    function shuffle(a){ for(let i=a.length-1;i>0;i--){ const j=Math.floor(Math.random()*(i+1)); [a[i],a[j]]=[a[j],a[i]]; } return a; }

    function renderTarget(){ targetEl.innerHTML = answer.map(ch=>"<span class='char'>"+ch+"</span>").join(""); }
    function setStatus(msg,good=false,bad=false){
      statusEl.textContent = msg||"";
      statusEl.classList.remove("ok","bad");
      if (good) statusEl.classList.add("ok");
      if (bad) statusEl.classList.add("bad");
    }

    function fillPool(){
      // Bias pool toward the current answer, then pad with decoys
      let pool = new Set();
      if (current) {
        for (const ch of current.jp) pool.add(ch);
      }
      const decoys = shuffle(ALL.slice());
      for (const ch of decoys){
        pool.add(ch);
        if (pool.size >= Math.min(40, ALL.length)) break; // keep pool manageable
      }
      const pieces = shuffle(Array.from(pool));
      poolEl.innerHTML = pieces.map(ch=>"<button class='piece' data-ch='"+ch+"'>"+ch+"</button>").join("");
    }

    function newWord(){
      current = choice(WORDS);
      answer = [];
      boardEl.textContent = current.en;
      romajiEl.textContent = "(" + current.romaji + ")";
      renderTarget();
      setStatus("Build the katakana word.");
      fillPool();
    }

    poolEl.addEventListener("click", e=>{
      const b = e.target.closest("button.piece"); if(!b) return;
      answer.push(b.getAttribute("data-ch"));
      renderTarget();
      setStatus("");
    });

    backBtn.addEventListener("click", ()=>{ answer.pop(); renderTarget(); });
    clearBtn.addEventListener("click", ()=>{ answer=[]; renderTarget(); setStatus("Cleared."); });
    shuffleBtn.addEventListener("click", ()=> fillPool());
    newBtn.addEventListener("click", ()=> newWord());

    checkBtn.addEventListener("click", ()=>{
      const guess = answer.join("");
      if (!current) return;
      if (guess === current.jp){
        setStatus("Correct! " + guess, true, false);
      } else {
        setStatus("Not yet. You made: " + guess + " (target: " + current.jp + ")", false, true);
      }
    });

    playBtn.addEventListener("click", ()=>{
      const guess = answer.join("");
      speakJa(guess || (current ? current.jp : ""));
    });

    // init
    newWord();
  })();
  </script>
</div>
"""))

html = tpl.safe_substitute(
    root_id=root_id,
    uid=uid,
    WORDS_JSON=json.dumps(WORDS, ensure_ascii=False),
    PIECES_JSON=json.dumps(PIECES, ensure_ascii=False),
)

display(HTML(html))


---